# Using Pretrained Models from Keras

In [ ]:
import tensorflow as tf


In [ ]:
res50=tf.keras.applications.ResNet50(weights="imagenet") #will load the weights of resNet that  trained on ImageNet dataset

102967424/102967424 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
from sklearn.datasets import load_sample_images
images = load_sample_images()["images"]
images_resized = tf.keras.layers.Resizing(height=224, width=224,crop_to_aspect_ratio=True)(images)

In [ ]:
 inputs = tf.keras.applications.resnet50.preprocess_input(images_resized) # preprocess_input = makes sure your images have the same normalization and scaling as the ones the model was originally trained on

In [ ]:
Y_proba = res50.predict(inputs) #predictions of two images
Y_proba.shape # (2,1000) two images and each one has 1000 prob because resnet is trained on 1000 class

1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step


(2, 1000)

In [ ]:
Y_proba[0].argmax()

np.int64(611)

In [ ]:
Y_proba[0].shape # 1000 propability class

(1000,)

In [ ]:
top_K = tf.keras.applications.resnet50.decode_predictions(Y_proba, top=3)
for image_index in range(len(images)):
    print(f"Image #{image_index}")
    for class_id, name, y_proba in top_K[image_index]:
        print(f"  {class_id} - {name:12s} {y_proba:.2%}")

35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Image #0
  n03598930 - jigsaw_puzzle 30.68%
  n02782093 - balloon      17.17%
  n03888257 - parachute    5.57%
Image #1
  n04209133 - shower_cap   34.37%
  n09229709 - bubble       11.41%
  n02782093 - balloon      9.46%


# Pretrained Models for Transfer Learning

In [ ]:
!pip install tensorflow-datasets


In [ ]:
import tensorflow_datasets as tfds
dataset, info = tfds.load("tf_flowers", as_supervised=True, with_info=True)
dataset_size = info.splits["train"].num_examples  # 3670
class_names = info.features["label"].names  # ["dandelion", "daisy", ...]
n_classes = info.features["label"].num_classes  # 5

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/tf_flowers/incomplete.DKCRDS_3.0.1/tf_flowers-train.tfrecord*...:   0%|   …

Dataset tf_flowers downloaded and prepared to /root/tensorflow_datasets/tf_flowers/3.0.1. Subsequent calls will reuse this data.


In [ ]:
print(dataset_size)
print(class_names)
print(n_classes)

3670
['dandelion', 'daisy', 'tulips', 'sunflowers', 'roses']
5


In [ ]:
test_set_raw, valid_set_raw, train_set_raw = tfds.load("tf_flowers",split=["train[:10%]", "train[10%:25%]", "train[25%:]"],as_supervised=True)

## data preprocessing

In [ ]:
batch_size = 32
preprocess = tf.keras.Sequential([
tf.keras.layers.Resizing(height=224, width=224, crop_to_aspect_ratio=True),
tf.keras.layers.Lambda(tf.keras.applications.xception.preprocess_input)])
train_set = train_set_raw.map(lambda X, y: (preprocess(X), y))
train_set = train_set.shuffle(1000, seed=42).batch(batch_size).prefetch(1)
valid_set = valid_set_raw.map(lambda X, y: (preprocess(X), y)).batch(batch_size)
test_set = test_set_raw.map(lambda X, y: (preprocess(X), y)).batch(batch_size)

## data augmentation

In [ ]:
 data_augmentation = tf.keras.Sequential([
 tf.keras.layers.RandomFlip(mode="horizontal", seed=42),
 tf.keras.layers.RandomRotation(factor=0.05, seed=42),
 tf.keras.layers.RandomContrast(factor=0.2, seed=42)
 ])

# transfer learning witth xecption model for classification

## build the xception model
### We have remove the top "include_top=False" because imgnet data contians 1000 but we have just 5 classes so we will add manually

In [ ]:
base_model = tf.keras.applications.xception.Xception(weights="imagenet",include_top=False)
avg = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
output = tf.keras.layers.Dense(n_classes, activation="softmax")(avg)
model = tf.keras.Model(inputs=base_model.input, outputs=output)

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
# freezing the pretrained layers
for layer in base_model.layers:
 layer.trainable = False

In [ ]:
 optimizer = tf.keras.optimizers.SGD(learning_rate=0.1, momentum=0.9)
 model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,metrics=["accuracy"])
 history = model.fit(train_set, validation_data=valid_set, epochs=3)

Epoch 1/3
86/86 ━━━━━━━━━━━━━━━━━━━━ 58s 382ms/step - accuracy: 0.6948 - loss: 1.0240 - val_accuracy: 0.8312 - val_loss: 0.6489
Epoch 2/3
86/86 ━━━━━━━━━━━━━━━━━━━━ 14s 154ms/step - accuracy: 0.9200 - loss: 0.3193 - val_accuracy: 0.8657 - val_loss: 0.6100
Epoch 3/3
86/86 ━━━━━━━━━━━━━━━━━━━━ 14s 162ms/step - accuracy: 0.9241 - loss: 0.2606 - val_accuracy: 0.8730 - val_loss: 0.5862


In [ ]:
for layer in base_model.layers[56:]:
  layer.trainable = True

In [ ]:
optimizer = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
metrics=["accuracy"])
history = model.fit(train_set, validation_data=valid_set, epochs=10)

Epoch 1/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 55s 416ms/step - accuracy: 0.8741 - loss: 0.3761 - val_accuracy: 0.8657 - val_loss: 0.6665
Epoch 2/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 63s 307ms/step - accuracy: 0.9823 - loss: 0.0632 - val_accuracy: 0.9093 - val_loss: 0.3033
Epoch 3/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 40s 301ms/step - accuracy: 0.9987 - loss: 0.0096 - val_accuracy: 0.9147 - val_loss: 0.2871
Epoch 4/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 27s 307ms/step - accuracy: 0.9974 - loss: 0.0122 - val_accuracy: 0.9129 - val_loss: 0.2950
Epoch 5/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 26s 300ms/step - accuracy: 0.9970 - loss: 0.0089 - val_accuracy: 0.9056 - val_loss: 0.3226
Epoch 6/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 41s 303ms/step - accuracy: 0.9970 - loss: 0.0071 - val_accuracy: 0.9129 - val_loss: 0.3164
Epoch 7/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 41s 306ms/step - accuracy: 0.9972 - loss: 0.0061 - val_accuracy: 0.9147 - val_loss: 0.2986
Epoch 8/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 41s 302ms/step - accuracy: 0.9991 - loss: 0.0060 - val_accu

# Transfer Learninig classification and object detection

In [ ]:
base_model = tf.keras.applications.xception.Xception(weights="imagenet",include_top=False)
avg = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
class_output = tf.keras.layers.Dense(n_classes, activation="softmax")(avg)

loc_output = tf.keras.layers.Dense(4)(avg) # 4 numbers (x_center, y_center, width, height)

model = tf.keras.Model(inputs=base_model.input,
outputs=[class_output, loc_output])

model.compile(loss=["sparse_categorical_crossentropy", "mse"],loss_weights=[0.8, 0.2],  # depends on what you care most about
optimizer=optimizer, metrics=["accuracy"])